# 🚀 Bike Sharing Demand - 데이터 준비

로컬의 설정 파일과 데이터를 S3에 업로드하여 모델링 준비를 완료합니다.

## 사전 준비
- Kaggle에서 `bike-sharing-demand` 데이터셋 다운로드
- `data/` 폴더에 아래 파일 배치:
  - `train.csv` (Kaggle train.csv를 train/validation으로 분리)
  - `validation.csv`
  - `test.csv` (Kaggle test.csv)

## S3 구조
```
# 설정 버킷 (conf)
s3://gs-retail-awesome-conf-us-east-1/dev/hjsong/bike-sharing-demand/baseline-lgbm-v1/
    ├── env.yml
    ├── meta.yml
    └── model.yml

# 데이터 버킷 (data)
s3://gs-retail-awesome-data-us-east-1/dev/hjsong/bike-sharing-demand/v1.0/data/
    ├── train.csv
    ├── validation.csv
    └── test.csv
```

## 1. 환경 설정

In [ ]:
import os
import yaml
import boto3
import pandas as pd
from pathlib import Path
from datetime import datetime
from botocore.exceptions import ClientError

BASE_DIR = Path.cwd()
CONF_DIR = BASE_DIR / 'conf'
DATA_DIR = BASE_DIR / 'data'

print(f"📁 Base Directory: {BASE_DIR}")
print(f"📁 Conf Directory: {CONF_DIR}")
print(f"📁 Data Directory: {DATA_DIR}")

## 2. 설정 파일 로드

In [ ]:
def load_yaml(filepath: Path) -> dict:
    with open(filepath, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

env_config   = load_yaml(CONF_DIR / 'env.yml')
meta_config  = load_yaml(CONF_DIR / 'meta.yml')
model_config = load_yaml(CONF_DIR / 'model.yml')

print("✅ 설정 파일 로드 완료")
print(f"   - env:        {env_config['env']}")
print(f"   - region:     {env_config['region']}")
print(f"   - user_id:    {meta_config['user_id']}")
print(f"   - project:    {meta_config['project']}")
print(f"   - experiment: {meta_config['experiment']}")
print(f"   - version:    {meta_config['version']}")

## 3. 데이터 분리 (Kaggle train.csv → train / validation)

In [ ]:
# Kaggle train.csv → 시간 기반 train / validation 분리
# (시계열 데이터이므로 랜덤 split 금지 - datetime 순 정렬 후 마지막 20%를 validation으로 사용)
# validation.csv가 이미 있으면 스킵합니다.

kaggle_train_path = DATA_DIR / 'train.csv'
train_out_path    = DATA_DIR / 'train.csv'
validation_path   = DATA_DIR / 'validation.csv'

if kaggle_train_path.exists() and not validation_path.exists():
    df = pd.read_csv(kaggle_train_path, parse_dates=['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)

    cutoff_ratio = model_config['data']['val_cutoff_ratio']  # 0.2
    cutoff_idx   = int(len(df) * (1 - cutoff_ratio))
    cutoff_date  = df.loc[cutoff_idx, 'datetime']

    train_df = df.iloc[:cutoff_idx]
    val_df   = df.iloc[cutoff_idx:]

    train_df.to_csv(train_out_path, index=False)
    val_df.to_csv(validation_path, index=False)

    print(f"✅ 시간 기반 데이터 분리 완료")
    print(f"   - cutoff 날짜: {cutoff_date}")
    print(f"   - train:      {len(train_df):,} rows  ({train_df['datetime'].min()} ~ {train_df['datetime'].max()})")
    print(f"   - validation: {len(val_df):,} rows  ({val_df['datetime'].min()} ~ {val_df['datetime'].max()})")
else:
    print("ℹ️  train.csv / validation.csv 이미 존재, 분리 스킵")

# 데이터 확인
for fname in ['train.csv', 'validation.csv', 'test.csv']:
    p = DATA_DIR / fname
    status = '✅' if p.exists() else '❌'
    size = f"{p.stat().st_size / 1024:.1f} KB" if p.exists() else 'missing'
    print(f"   {status} {fname} ({size})")

## 4. S3 경로 구성

In [ ]:
ENV        = env_config['env']
REGION     = env_config['region']
USER_ID    = meta_config['user_id']
PROJECT    = meta_config['project']
EXPERIMENT = meta_config['experiment']
VERSION    = meta_config['version']

CONF_BUCKET  = env_config['s3']['conf_bucket']
DATA_BUCKET  = env_config['s3']['data_bucket']
MODEL_BUCKET = env_config['s3']['model_bucket']
ALL_BUCKETS  = [CONF_BUCKET, DATA_BUCKET, MODEL_BUCKET]

CONF_PREFIX = f"{ENV}/{USER_ID}/{PROJECT}/{EXPERIMENT}"
DATA_PREFIX = f"{ENV}/{USER_ID}/{PROJECT}/{VERSION}"

S3_CONF_PATH  = f"s3://{CONF_BUCKET}/{CONF_PREFIX}/"
S3_DATA_PATH  = f"s3://{DATA_BUCKET}/{DATA_PREFIX}/"

print("📦 S3 경로 구성")
print(f"   CONF:  {S3_CONF_PATH}")
print(f"   DATA:  {S3_DATA_PATH}")

## 5. S3 클라이언트 초기화 및 버킷 확인

In [ ]:
s3_client = boto3.client('s3', region_name=REGION)

def bucket_exists(bucket_name: str) -> bool:
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        return True
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == '404':
            return False
        elif error_code == '403':
            print(f"   ⚠️  {bucket_name}: 접근 권한 없음 (버킷은 존재)")
            return True
        raise e

def create_bucket(bucket_name: str, region: str) -> bool:
    try:
        if region == 'us-east-1':
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3_client.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={'LocationConstraint': region}
            )
        return True
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code in ('BucketAlreadyOwnedByYou', 'BucketAlreadyExists'):
            return True
        print(f"   ❌ {bucket_name}: 생성 실패 - {e}")
        return False

def ensure_bucket_exists(bucket_name: str, region: str) -> bool:
    if bucket_exists(bucket_name):
        print(f"   ✅ {bucket_name} (존재)")
        return True
    print(f"   🆕 {bucket_name} 생성 중...")
    success = create_bucket(bucket_name, region)
    if success:
        print(f"   ✅ {bucket_name} (생성 완료)")
    return success

print("🪣 S3 버킷 확인")
print("=" * 60)
for bucket in ALL_BUCKETS:
    ensure_bucket_exists(bucket, REGION)
print("=" * 60)

## 6. 설정 파일 업로드 (conf 버킷)

In [ ]:
def upload_file_to_s3(local_path: Path, bucket: str, s3_key: str) -> bool:
    try:
        s3_client.upload_file(str(local_path), bucket, s3_key)
        return True
    except Exception as e:
        print(f"❌ 업로드 실패: {local_path} -> {e}")
        return False

CONF_FILES = ['env.yml', 'meta.yml', 'model.yml']

print(f"📤 설정 파일 업로드")
print(f"   대상: {S3_CONF_PATH}")
print(f"   " + "-" * 50)

for filename in CONF_FILES:
    local_path = CONF_DIR / filename
    s3_key = f"{CONF_PREFIX}/{filename}"
    if local_path.exists():
        success = upload_file_to_s3(local_path, CONF_BUCKET, s3_key)
        status = '✅' if success else '❌'
        print(f"   {status} {filename}")
    else:
        print(f"   ⚠️  {filename} 없음 (스킵)")

## 7. 데이터 파일 업로드 (data 버킷)

In [ ]:
DATA_FILES = ['train.csv', 'validation.csv', 'test.csv']

print(f"📤 데이터 파일 업로드")
print(f"   대상: {S3_DATA_PATH}data/")
print(f"   " + "-" * 50)

for filename in DATA_FILES:
    local_path = DATA_DIR / filename
    s3_key = f"{DATA_PREFIX}/data/{filename}"
    if local_path.exists():
        size_str = f"{local_path.stat().st_size / 1024:.1f} KB"
        success = upload_file_to_s3(local_path, DATA_BUCKET, s3_key)
        status = '✅' if success else '❌'
        print(f"   {status} {filename} ({size_str})")
    else:
        print(f"   ⚠️  {filename} 없음 (스킵)")

## 8. 업로드 결과 요약

In [ ]:
print("\n" + "=" * 60)
print("📋 데이터 준비 완료 요약")
print("=" * 60)
print(f"""
🏷️  프로젝트 정보
    - Project:    {PROJECT}
    - Experiment: {EXPERIMENT}
    - Version:    {VERSION}
    - User:       {USER_ID}
    - Env:        {ENV}

📦 S3 경로
    - Conf: {S3_CONF_PATH}
    - Data: {S3_DATA_PATH}data/

🚀 다음 단계
    modeling/ 폴더의 load_conf_datasets.ipynb 실행
    → S3에서 conf/, data/ 다운로드 후 modeling.ipynb 실행
""")
print(f"⏰ 완료 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)